# NHAMCS 2018–2022 5-Class Acuity: Comparative Analysis of Random Forest with KFDA vs Non-KFDA Inputs (`models/train_distant_analysis.ipynb`)

Trains on the official CDC **National Hospital Ambulatory Medical Care Survey (NHAMCS 2018–2022)** dataset (**`datasets/nhamcs_2018_2022.csv`**), benchmarking and comparing **Random Forest fitted with KFDA non-linear projection inputs versus Non-KFDA (Raw Feature) inputs** across all **5 individual triage levels (`IMMEDR 1–5`)**:
1. **`IMMEDR 1: Immediate`** ($y=0$): Resuscitation required immediately ($846$ visits, $\sim 1.46\%$).
2. **`IMMEDR 2: Emergent`** ($y=1$): Condition requiring emergent care within 1–14 minutes ($8,597$ visits, $\sim 14.79\%$).
3. **`IMMEDR 3: Urgent`** ($y=2$): Condition requiring urgent care within 15–60 minutes ($29,568$ visits, $\sim 50.87\%$).
4. **`IMMEDR 4: Semi-urgent`** ($y=3$): Condition requiring semi-urgent care within 1–2 hours ($16,715$ visits, $\sim 28.76\%$).
5. **`IMMEDR 5: Nonurgent`** ($y=4$): Condition requiring nonurgent care within 2–24 hours ($2,398$ visits, $\sim 4.13\%$).

```mermaid
flowchart TD
    Raw["Raw NHAMCS Arrival Features X in R^7 (58,124 Visits)"] --> Imputer["SimpleImputer(strategy='median') & StandardScaler()"]
    Imputer --> Eng["Clinical Composite Engineering (+4 Non-Linear Indices: SI, PP, ROX, Age-SI)"]
    
    Eng --> BranchA["Branch A: Non-KFDA (Raw 11 Features in R^11)"]
    BranchA --> RFRaw["Model A: Balanced Random Forest (Raw Features)"]
    
    Eng --> BranchB["Branch B: KFDA Non-Linear Projection"]
    BranchB --> Nystroem["Nystroem RBF Kernel Mapping phi(X) in R^600"]
    Nystroem --> KFDA["Multi-Class LDA (R^600 -> R^4 Canonical Coordinates)"]
    KFDA --> RFKFDA["Model B: Balanced Random Forest (KFDA Features)"]
    
    RFRaw & RFKFDA --> Comp["Side-by-Side Holdout Test Evaluation & Benchmark Comparison"]
```

### 🔬 Benchmark Comparison Matrix
| Feature Set / Pipeline | Input Dimension | Transformation / Kernel | Target Classifier |
|---|---|---|---|
| **Model A: Non-KFDA Baseline** | $\mathbb{R}^{11}$ | Direct Z-score Standardized Features | `RandomForestClassifier(n_estimators=300, class_weight='balanced')` |
| **Model B: KFDA Pipeline** | $\mathbb{R}^{4}$ | Nystroem RBF Kernel ($\gamma=0.15, m=600$) + Multi-Class LDA | `RandomForestClassifier(n_estimators=300, class_weight='balanced')` |

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Load NHAMCS Dataset, Clean Missing Codes & Apply Train/Test Split
# ---------------------------------------------------------------------------
import os, json, pickle, warnings, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.kernel_approximation import Nystroem
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, recall_score,
    precision_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
ROOT = ".." if os.path.basename(os.getcwd()) == "models" else "."
data_path = f"{ROOT}/datasets/nhamcs_2018_2022.csv"
if not os.path.exists(data_path) and os.path.exists("datasets/nhamcs_2018_2022.csv"):
    data_path = "datasets/nhamcs_2018_2022.csv"
elif not os.path.exists(data_path) and os.path.exists("nhamcs_2018_2022.csv"):
    data_path = "nhamcs_2018_2022.csv"
elif not os.path.exists(data_path) and os.path.exists(f"{ROOT}/datasets/ed2022.csv"):
    data_path = f"{ROOT}/datasets/ed2022.csv"

print(f"Loading NHAMCS Dataset from: {data_path} ...")
raw_df = pd.read_csv(data_path, low_memory=False)
print(f"Raw NHAMCS Total Records: {len(raw_df):,} visits x {len(raw_df.columns)} variables")

# Filter ONLY rows where IMMEDR is in 1..5 (Ignore -9, -8, 0, 7)
valid_mask = raw_df["IMMEDR"].isin([1, 2, 3, 4, 5])
df_filtered = raw_df[valid_mask].copy()
print(f"Filtered Cohort (IMMEDR in 1..5): {len(df_filtered):,} visits ({len(df_filtered)/len(raw_df)*100:.2f}% of total)")

# Extract Features
feature_cols = ["AGE", "SEX", "PULSE", "RESPR", "BPSYS", "BPDIAS", "POPCT"]
X_raw_df = df_filtered[feature_cols].copy()

# Clean special missing / unmeasured codes to np.nan for median imputation
# -9 = Blank, 998 = Doppler / Palpation
X_raw_df["PULSE"]  = X_raw_df["PULSE"].replace([-9, 998], np.nan)
X_raw_df["RESPR"]  = X_raw_df["RESPR"].replace([-9], np.nan)
X_raw_df["BPSYS"]  = X_raw_df["BPSYS"].replace([-9], np.nan)
X_raw_df["BPDIAS"] = X_raw_df["BPDIAS"].replace([-9, 998], np.nan)
X_raw_df["POPCT"]  = X_raw_df["POPCT"].replace([-9], np.nan)

# Recode SEX: 1=Female -> 0, 2=Male -> 1
X_raw_df["SEX"] = (X_raw_df["SEX"] == 2).astype(float)

RAW_FEATURE_NAMES = ["age", "gender_male", "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr", "triage_vital_o2"]
X_raw = X_raw_df[["AGE", "SEX", "PULSE", "BPSYS", "BPDIAS", "RESPR", "POPCT"]].values

# Target 5-Class Mapping: IMMEDR 1=0, 2=1, 3=2, 4=3, 5=4
immedr_vals = df_filtered["IMMEDR"].values
y_all = (immedr_vals - 1).astype(int)
TIER_LABELS = [
    "IMMEDR 1: Immediate",
    "IMMEDR 2: Emergent",
    "IMMEDR 3: Urgent",
    "IMMEDR 4: Semi-urgent",
    "IMMEDR 5: Nonurgent"
]

print("=" * 80)
print("  5-CLASS IMMEDR TRIAGE DISTRIBUTION (NHAMCS 2018-2022)")
print("=" * 80)
for c, lbl in enumerate(TIER_LABELS):
    count_c = np.sum(y_all == c)
    print(f"  * Level {c+1} [{lbl:<22}]: {count_c:>6,} encounters ({count_c/len(y_all)*100:.2f}%)")
print("=" * 80 + chr(10))

# Stratified 70% Train / 15% Validation / 15% Test Split
itr, itmp = train_test_split(np.arange(len(y_all)), test_size=0.30, stratify=y_all, random_state=42)
iva, ite  = train_test_split(itmp, test_size=0.50, stratify=y_all[itmp], random_state=42)

y_train, y_val, y_test = y_all[itr], y_all[iva], y_all[ite]

# Median Imputation fitted strictly on Training partition
imputer   = SimpleImputer(strategy="median")
X_tr_imp  = imputer.fit_transform(X_raw[itr])
X_val_imp = imputer.transform(X_raw[iva])
X_te_imp  = imputer.transform(X_raw[ite])

# Function to add Non-Linear Clinical Composite Indices
def add_clinical_composites(X_arr):
    age  = X_arr[:, 0]
    hr   = X_arr[:, 2]
    sbp  = X_arr[:, 3]
    dbp  = X_arr[:, 4]
    rr   = X_arr[:, 5]
    o2   = X_arr[:, 6]
    
    si     = hr / np.clip(sbp, 40.0, 300.0)             # Shock Index: HR / SBP
    pp     = np.clip(sbp - dbp, 1.0, 200.0)             # Pulse Pressure: SBP - DBP
    rox    = o2 / np.clip(rr, 4.0, 120.0)               # ROX Index: SpO2 / RR
    age_si = age * si                                   # Age-Adjusted Shock Index
    return np.column_stack([X_arr, si, pp, rox, age_si])

X_tr_comp  = add_clinical_composites(X_tr_imp)
X_val_comp = add_clinical_composites(X_val_imp)
X_te_comp  = add_clinical_composites(X_te_imp)

ALL_FEATURE_NAMES = RAW_FEATURE_NAMES + ["shock_index", "pulse_pressure", "rox_index", "age_shock_index"]

# Standard z-score scaling
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_comp)
X_val   = scaler.transform(X_val_comp)
X_test  = scaler.transform(X_te_comp)

print(f"Partition Dimensions:")
print(f"  * Training Set  : {X_train.shape[0]:,} visits, {X_train.shape[1]} engineered features (70%)")
print(f"  * Validation Set: {X_val.shape[0]:,} visits (15%)")
print(f"  * Holdout Test  : {X_test.shape[0]:,} visits (15%)")

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Train Model A — Baseline Balanced Random Forest (Non-KFDA Inputs in R^11)
# ---------------------------------------------------------------------------
print("=" * 80)
print("  TRAINING MODEL A: NON-KFDA BALANCED RANDOM FOREST (RAW FEATURES IN R^11)")
print("=" * 80)

rf_raw_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rf_raw_model.fit(X_train, y_train)
print(f"✓ Model A (Non-KFDA RF) Training Completed in {time.time()-t0:.1f}s!")

# Quick Validation Check for Model A
pred_val_raw = rf_raw_model.predict(X_val)
val_bacc_raw = balanced_accuracy_score(y_val, pred_val_raw)
val_rec0_raw = recall_score(y_val, pred_val_raw, labels=[0], average=None, zero_division=0)[0]
print(f"Validation Check [Model A: Non-KFDA RF]:")
print(f"  * Macro Balanced Accuracy      : {val_bacc_raw*100:.2f}%")
print(f"  * IMMEDR 1 (Immediate) Recall  : {val_rec0_raw*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Train Model B — KFDA Non-Linear Projection + Balanced Random Forest (R^4)
# ---------------------------------------------------------------------------
print("=" * 80)
print("  TRAINING MODEL B: KFDA PROJECTION + BALANCED RANDOM FOREST (R^4)")
print("=" * 80)

# 1. Nystroem Non-Linear RBF Kernel Landmark Approximation (R^11 -> R^600)
nystroem_kfda = Nystroem(
    kernel='rbf',
    gamma=0.15,
    n_components=600,
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
Phi_train = nystroem_kfda.fit_transform(X_train)
Phi_val   = nystroem_kfda.transform(X_val)
Phi_test  = nystroem_kfda.transform(X_test)
print(f"✓ Nystroem Kernel Mapping fitted in {time.time()-t0:.1f}s (Shape: {Phi_train.shape})")

# 2. Multi-Class Linear Discriminant Analysis on Kernel Space (KFDA)
# Extracts 4 canonical discriminant coordinates [z1, z2, z3, z4] for C=5 classes
kfda_lda = LinearDiscriminantAnalysis(n_components=4)

t0 = time.time()
Z_train = kfda_lda.fit_transform(Phi_train, y_train)
Z_val   = kfda_lda.transform(Phi_val)
Z_test  = kfda_lda.transform(Phi_test)
print(f"✓ KFDA Fisher Projection fitted in {time.time()-t0:.1f}s")
print(f"  Explained Variance Ratios per Coordinate: {np.round(kfda_lda.explained_variance_ratio_, 4)}")

# 3. Fit Balanced Random Forest on 4D KFDA coordinates
rf_kfda_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=14,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

t0 = time.time()
rf_kfda_model.fit(Z_train, y_train)
print(f"✓ Model B (KFDA RF) Training Completed in {time.time()-t0:.1f}s!")

# Quick Validation Check for Model B
pred_val_kfda = rf_kfda_model.predict(Z_val)
val_bacc_kfda = balanced_accuracy_score(y_val, pred_val_kfda)
val_rec0_kfda = recall_score(y_val, pred_val_kfda, labels=[0], average=None, zero_division=0)[0]
print(f"\nValidation Check [Model B: KFDA RF]:")
print(f"  * Macro Balanced Accuracy      : {val_bacc_kfda*100:.2f}%")
print(f"  * IMMEDR 1 (Immediate) Recall  : {val_rec0_kfda*100:.2f}%")

In [ ]:
# ---------------------------------------------------------------------------
# Step 4: Comprehensive Holdout Test Evaluation — Non-KFDA RF vs KFDA RF
# ---------------------------------------------------------------------------
# Predictions on Holdout Test Set (8,719 visits)
pred_test_raw  = rf_raw_model.predict(X_test)
p_test_raw     = rf_raw_model.predict_proba(X_test)

pred_test_kfda = rf_kfda_model.predict(Z_test)
p_test_kfda    = rf_kfda_model.predict_proba(Z_test)

y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3, 4])

# Helper function to compute full 5-class metrics
def evaluate_model(y_true, y_pred, y_prob, model_name):
    acc     = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    mf1     = f1_score(y_true, y_pred, average='macro', zero_division=0)
    wf1     = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    auc_ovr = roc_auc_score(y_test_bin, y_prob, average='macro', multi_class='ovr')
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3, 4])
    rec_per = recall_score(y_true, y_pred, average=None, zero_division=0)
    prec_per = precision_score(y_true, y_pred, average=None, zero_division=0)
    f1_per   = f1_score(y_true, y_pred, average=None, zero_division=0)
    
    spec_per = []
    for c in range(5):
        tp = cm[c, c]
        fn = cm[c, :].sum() - tp
        fp = cm[:, c].sum() - tp
        tn = cm.sum() - tp - fn - fp
        spec_per.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)
    
    auc_per = [roc_auc_score(y_test_bin[:, c], y_prob[:, c]) for c in range(5)]
    
    return {
        'name': model_name,
        'acc': acc, 'bal_acc': bal_acc, 'macro_spec': np.mean(spec_per),
        'macro_f1': mf1, 'weighted_f1': wf1, 'auc_ovr': auc_ovr,
        'rec_per': rec_per, 'spec_per': spec_per, 'prec_per': prec_per,
        'f1_per': f1_per, 'auc_per': auc_per, 'cm': cm
    }

res_raw  = evaluate_model(y_test, pred_test_raw, p_test_raw, 'Non-KFDA RF (Raw Features)')
res_kfda = evaluate_model(y_test, pred_test_kfda, p_test_kfda, 'KFDA + Balanced RF')

# ---------------------------------------------------------------------------
# 1. Overall Macro Metric Comparison Table
# ---------------------------------------------------------------------------
overall_comp_df = pd.DataFrame([
    {
        'Metric': 'Overall Accuracy',
        'Model A (Non-KFDA RF)': f"{res_raw['acc']*100:.2f}%",
        'Model B (KFDA RF)': f"{res_kfda['acc']*100:.2f}%",
        'Delta (KFDA - Raw)': f"{(res_kfda['acc'] - res_raw['acc'])*100:+.2f}%"
    },
    {
        'Metric': 'Macro Balanced Accuracy',
        'Model A (Non-KFDA RF)': f"{res_raw['bal_acc']*100:.2f}%",
        'Model B (KFDA RF)': f"{res_kfda['bal_acc']*100:.2f}%",
        'Delta (KFDA - Raw)': f"{(res_kfda['bal_acc'] - res_raw['bal_acc'])*100:+.2f}%"
    },
    {
        'Metric': 'Macro Specificity (TNR)',
        'Model A (Non-KFDA RF)': f"{res_raw['macro_spec']*100:.2f}%",
        'Model B (KFDA RF)': f"{res_kfda['macro_spec']*100:.2f}%",
        'Delta (KFDA - Raw)': f"{(res_kfda['macro_spec'] - res_raw['macro_spec'])*100:+.2f}%"
    },
    {
        'Metric': 'Macro ROC-AUC (OvR)',
        'Model A (Non-KFDA RF)': f"{res_raw['auc_ovr']:.4f}",
        'Model B (KFDA RF)': f"{res_kfda['auc_ovr']:.4f}",
        'Delta (KFDA - Raw)': f"{res_kfda['auc_ovr'] - res_raw['auc_ovr']:+.4f}"
    },
    {
        'Metric': 'Macro F1-Score',
        'Model A (Non-KFDA RF)': f"{res_raw['macro_f1']:.4f}",
        'Model B (KFDA RF)': f"{res_kfda['macro_f1']:.4f}",
        'Delta (KFDA - Raw)': f"{res_kfda['macro_f1'] - res_raw['macro_f1']:+.4f}"
    },
    {
        'Metric': 'Weighted F1-Score',
        'Model A (Non-KFDA RF)': f"{res_raw['weighted_f1']:.4f}",
        'Model B (KFDA RF)': f"{res_kfda['weighted_f1']:.4f}",
        'Delta (KFDA - Raw)': f"{res_kfda['weighted_f1'] - res_raw['weighted_f1']:+.4f}"
    }
])

print("=" * 95)
print("     HOLDOUT TEST OVERALL BENCHMARK: NON-KFDA RF vs KFDA RF (NHAMCS 2018-2022)")
print("=" * 95)
print(overall_comp_df.to_string(index=False))
print("=" * 95 + chr(10))

# ---------------------------------------------------------------------------
# 2. Per-Class Detailed Comparison Table
# ---------------------------------------------------------------------------
per_class_rows = []
for c, lbl in enumerate(TIER_LABELS):
    per_class_rows.append({
        'Triage_Level': lbl,
        'True_Count': int(np.sum(y_test == c)),
        'Raw_Recall': f"{res_raw['rec_per'][c]*100:.2f}%",
        'KFDA_Recall': f"{res_kfda['rec_per'][c]*100:.2f}%",
        'Delta_Recall': f"{(res_kfda['rec_per'][c] - res_raw['rec_per'][c])*100:+.2f}%",
        'Raw_Spec': f"{res_raw['spec_per'][c]*100:.2f}%",
        'KFDA_Spec': f"{res_kfda['spec_per'][c]*100:.2f}%",
        'Raw_ROC_AUC': round(res_raw['auc_per'][c], 4),
        'KFDA_ROC_AUC': round(res_kfda['auc_per'][c], 4),
        'Delta_AUC': round(res_kfda['auc_per'][c] - res_raw['auc_per'][c], 4)
    })

per_class_df = pd.DataFrame(per_class_rows)
print("PER-CLASS RECALL & ROC-AUC COMPARISON:")
print(per_class_df.to_string(index=False))
print("=" * 95 + chr(10))

# Export Full Comparison CSV
reports_dir = f'{ROOT}/reports'
os.makedirs(reports_dir, exist_ok=True)
report_file = os.path.join(reports_dir, 'nhamcs_5class_kfda_vs_raw_rf_comparison.csv')
per_class_df.to_csv(report_file, index=False)
print(f"✓ Full benchmark comparison report saved to: {report_file}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Side-by-Side 5x5 Confusion Matrices (Non-KFDA vs KFDA)
# ---------------------------------------------------------------------------
plots_dir = f"{ROOT}/plots/distant_analysis"
os.makedirs(plots_dir, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))
short_labels = ['1: Imm', '2: Emerg', '3: Urgent', '4: Semi-urg', '5: Nonurg']

# Left: Model A (Non-KFDA RF)
cm_raw = res_raw['cm']
cm_raw_norm = cm_raw.astype('float') / cm_raw.sum(axis=1)[:, np.newaxis]
annot_raw = np.empty_like(cm_raw, dtype=object)
for i in range(5):
    for j in range(5):
        annot_raw[i, j] = f"{cm_raw[i, j]:,}\n({cm_raw_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_raw_norm, annot=annot_raw, fmt='', cmap='Blues', cbar=True, ax=axes[0],
    vmin=0, vmax=1, xticklabels=short_labels, yticklabels=short_labels
)
axes[0].set_title(
    f"Model A: Non-KFDA Random Forest (Raw Features in R^11)\n"
    f"Balanced Acc: {res_raw['bal_acc']*100:.2f}% | Macro ROC-AUC: {res_raw['auc_ovr']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[0].set_xlabel("Predicted IMMEDR Level", fontsize=11, fontweight='bold')
axes[0].set_ylabel("True IMMEDR Level", fontsize=11, fontweight='bold')

# Right: Model B (KFDA + RF)
cm_kfda = res_kfda['cm']
cm_kfda_norm = cm_kfda.astype('float') / cm_kfda.sum(axis=1)[:, np.newaxis]
annot_kfda = np.empty_like(cm_kfda, dtype=object)
for i in range(5):
    for j in range(5):
        annot_kfda[i, j] = f"{cm_kfda[i, j]:,}\n({cm_kfda_norm[i, j]*100:.1f}%)"

sns.heatmap(
    cm_kfda_norm, annot=annot_kfda, fmt='', cmap='Greens', cbar=True, ax=axes[1],
    vmin=0, vmax=1, xticklabels=short_labels, yticklabels=short_labels
)
axes[1].set_title(
    f"Model B: KFDA + Balanced Random Forest (Canonical Space R^4)\n"
    f"Balanced Acc: {res_kfda['bal_acc']*100:.2f}% | Macro ROC-AUC: {res_kfda['auc_ovr']:.4f}",
    fontsize=11.5, fontweight='bold', pad=12
)
axes[1].set_xlabel("Predicted IMMEDR Level", fontsize=11, fontweight='bold')
axes[1].set_ylabel("True IMMEDR Level", fontsize=11, fontweight='bold')

plt.tight_layout()
cm_comp_path = os.path.join(plots_dir, "nhamcs_5class_side_by_side_confusion_matrix.png")
plt.savefig(cm_comp_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Side-by-side Confusion Matrix comparison saved to: {cm_comp_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 6: Per-Class Metric Comparison Bar Charts (Non-KFDA vs KFDA)
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

x_pos = np.arange(5)
width = 0.35

# Subplot 1: Sensitivity / Recall per Level
axes[0].bar(x_pos - width/2, res_raw['rec_per'] * 100, width, label='Model A: Non-KFDA RF', color='#1f77b4', alpha=0.85)
axes[0].bar(x_pos + width/2, res_kfda['rec_per'] * 100, width, label='Model B: KFDA RF', color='#2ca02c', alpha=0.85)
axes[0].set_title('Sensitivity (Recall) per IMMEDR Triage Level', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Triage Level', fontsize=10, fontweight='bold')
axes[0].set_ylabel('Recall (%)', fontsize=10, fontweight='bold')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(['1: Imm', '2: Emerg', '3: Urgent', '4: Semi-urg', '5: Nonurg'])
axes[0].grid(True, linestyle='--', alpha=0.4)
axes[0].legend(loc='upper right', fontsize=9)

# Subplot 2: One-vs-Rest ROC-AUC per Level
axes[1].bar(x_pos - width/2, res_raw['auc_per'], width, label='Model A: Non-KFDA RF', color='#1f77b4', alpha=0.85)
axes[1].bar(x_pos + width/2, res_kfda['auc_per'], width, label='Model B: KFDA RF', color='#2ca02c', alpha=0.85)
axes[1].set_title('One-vs-Rest ROC-AUC per IMMEDR Triage Level', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Triage Level', fontsize=10, fontweight='bold')
axes[1].set_ylabel('ROC-AUC Score', fontsize=10, fontweight='bold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(['1: Imm', '2: Emerg', '3: Urgent', '4: Semi-urg', '5: Nonurg'])
axes[1].set_ylim([0.4, 0.9])
axes[1].grid(True, linestyle='--', alpha=0.4)
axes[1].legend(loc='upper right', fontsize=9)

plt.tight_layout()
metric_comp_path = os.path.join(plots_dir, "nhamcs_5class_kfda_vs_raw_metric_comparison.png")
plt.savefig(metric_comp_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Metric comparison chart saved to: {metric_comp_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 7: KFDA Discriminant Manifold vs Raw Features Decision Boundary Plots
# ---------------------------------------------------------------------------
palette = {
    0: '#d62728', # Red (Immediate)
    1: '#ff7f0e', # Orange (Emergent)
    2: '#bcbd22', # Yellow-Green (Urgent)
    3: '#1f77b4', # Blue (Semi-urgent)
    4: '#2ca02c'  # Green (Nonurgent)
}

fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Panel 1: Raw Overlapping Vitals (Shock Index vs Pulse Pressure)
ax1 = axes[0, 0]
for c in range(5):
    c_mask = (y_test == c)
    ax1.scatter(
        X_test[c_mask, 7], X_test[c_mask, 8],
        c=palette[c], label=TIER_LABELS[c], alpha=0.4 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax1.set_title("Raw Overlapping Vitals: Shock Index vs Pulse Pressure", fontsize=11, fontweight='bold')
ax1.set_xlabel("Standardized Shock Index (SI)", fontsize=10)
ax1.set_ylabel("Standardized Pulse Pressure (PP)", fontsize=10)
ax1.grid(True, linestyle='--', alpha=0.3)
ax1.legend(loc='upper right', fontsize=8)

# Panel 2: Raw Overlapping Vitals (ROX Index vs Heart Rate)
ax2 = axes[0, 1]
for c in range(5):
    c_mask = (y_test == c)
    ax2.scatter(
        X_test[c_mask, 9], X_test[c_mask, 2],
        c=palette[c], label=TIER_LABELS[c], alpha=0.4 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax2.set_title("Raw Overlapping Vitals: ROX Index vs Heart Rate", fontsize=11, fontweight='bold')
ax2.set_xlabel("Standardized ROX Index (SpO2 / RR)", fontsize=10)
ax2.set_ylabel("Standardized Heart Rate (PULSE)", fontsize=10)
ax2.grid(True, linestyle='--', alpha=0.3)

# Panel 3: 2D KFDA Canonical Coordinates (z1 vs z2)
ax3 = axes[1, 0]
for c in range(5):
    c_mask = (y_test == c)
    ax3.scatter(
        Z_test[c_mask, 0], Z_test[c_mask, 1],
        c=palette[c], label=TIER_LABELS[c], alpha=0.45 if c > 0 else 0.9,
        s=14 if c > 0 else 35, edgecolors='none'
    )
ax3.set_title("KFDA Projection: 2D Fisher Discriminant Coordinates (z1 vs z2)", fontsize=11, fontweight='bold')
ax3.set_xlabel(f"KFDA Coordinate z1 ({kfda_lda.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=10)
ax3.set_ylabel(f"KFDA Coordinate z2 ({kfda_lda.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=10)
ax3.grid(True, linestyle='--', alpha=0.3)

# Panel 4: 2D KFDA Space with 5-Class Decision Contours
ax4 = axes[1, 1]
x_min, x_max = Z_test[:, 0].min() - 0.5, Z_test[:, 0].max() + 0.5
y_min, y_max = Z_test[:, 1].min() - 0.5, Z_test[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))

rf_2d = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf_2d.fit(Z_train[:, :2], y_train)
grid_preds = rf_2d.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

cmap_light = plt.matplotlib.colors.ListedColormap(['#ffcccc', '#ffe0b2', '#fff9c4', '#bbdefb', '#c8e6c9'])
ax4.contourf(xx, yy, grid_preds, alpha=0.4, cmap=cmap_light)

for c in range(5):
    c_mask = (y_test == c)
    ax4.scatter(
        Z_test[c_mask, 0], Z_test[c_mask, 1],
        c=palette[c], label=TIER_LABELS[c], alpha=0.4 if c > 0 else 0.85,
        s=12 if c > 0 else 30, edgecolors='none'
    )
ax4.set_title("5-Class Random Forest Decision Boundary Contours on KFDA Space", fontsize=11, fontweight='bold')
ax4.set_xlabel(f"KFDA Coordinate z1", fontsize=10)
ax4.set_ylabel(f"KFDA Coordinate z2", fontsize=10)
ax4.grid(True, linestyle='--', alpha=0.3)

plt.tight_layout()
bound_plot_path = os.path.join(plots_dir, "nhamcs_5class_kfda_decision_boundaries.png")
plt.savefig(bound_plot_path, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Decision boundary plots saved to: {bound_plot_path}")

In [ ]:
# ---------------------------------------------------------------------------
# Step 8: Export Production Artifact Bundle & Metadata Manifest
# ---------------------------------------------------------------------------
deploy_dir = f'{ROOT}/deploy'
os.makedirs(deploy_dir, exist_ok=True)

bundle = {
    'imputer': imputer,
    'scaler': scaler,
    'rf_raw_model': rf_raw_model,
    'nystroem_kfda': nystroem_kfda,
    'kfda_lda': kfda_lda,
    'rf_kfda_model': rf_kfda_model,
    'raw_features': RAW_FEATURE_NAMES,
    'all_features': ALL_FEATURE_NAMES,
    'tier_labels': TIER_LABELS
}

bundle_file = os.path.join(deploy_dir, 'nhamcs_5class_kfda_vs_raw_bundle.pkl')
with open(bundle_file, 'wb') as f:
    pickle.dump(bundle, f)

manifest = dict(
    pipeline='NHAMCS_5Class_KFDA_vs_Raw_Random_Forest_Comparison',
    dataset='datasets/nhamcs_2018_2022.csv',
    target='IMMEDR (1 to 5)',
    n_classes=5,
    tier_labels=TIER_LABELS,
    raw_features=RAW_FEATURE_NAMES,
    engineered_features=ALL_FEATURE_NAMES,
    total_valid_samples=len(y_all),
    holdout_test_samples=len(y_test),
    model_a_non_kfda=dict(
        overall_accuracy=round(res_raw['acc'], 4),
        macro_balanced_accuracy=round(res_raw['bal_acc'], 4),
        macro_specificity=round(res_raw['macro_spec'], 4),
        macro_roc_auc_ovr=round(res_raw['auc_ovr'], 4),
        macro_f1=round(res_raw['macro_f1'], 4)
    ),
    model_b_kfda=dict(
        overall_accuracy=round(res_kfda['acc'], 4),
        macro_balanced_accuracy=round(res_kfda['bal_acc'], 4),
        macro_specificity=round(res_kfda['macro_spec'], 4),
        macro_roc_auc_ovr=round(res_kfda['auc_ovr'], 4),
        macro_f1=round(res_kfda['macro_f1'], 4)
    ),
    per_class_comparison=per_class_rows
)

manifest_file = os.path.join(deploy_dir, 'nhamcs_5class_kfda_vs_raw_manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Deployment Bundle  : {bundle_file}")
print(f"✓ Deployment Manifest: {manifest_file}")